# run the initial stuff

In [ ]:
import os
import sys
import time
import sympy as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import norm
from scipy.integrate import simpson
from scipy.optimize import minimize

from scipy import stats
from numpy.polynomial import Polynomial, Chebyshev

In [ ]:
for filename in ['base_waves_class_library',
                 'moments_function_library',
                 'polyfit_function_library',
                 'scatter_function_library',
                 'gmm_library']:
    !jupyter nbconvert --to python '/classes/{filename}.ipynb'

In [ ]:
import base_waves_class_library as bwl
import moments_function_library as mfl
import polyfit_function_library as pfl
import scatter_function_library as sfl
import gmm_library              as gmm

# Database

### Delta and GMM Databases

In [ ]:
moment_names    = ['mean', 'variance', 'std', 'skew', 'kurtosis', 'inv_kurt', 'hyperskewness', 'hyperkurtosis']
gmm_names       = ['loc_1', 'loc_2', 'loc_3', 'amp_1', 'amp_2', 'amp_3', 'mu1', 'mu2', 'sigma', 'w1'] + moment_names
cols            = ['loc_1', 'loc_2', 'loc_3', 'amp_1', 'amp_2', 'amp_3'] + moment_names
print(cols)

num_records     = 0
mid_low         = -0.1
mid_high        = 0.1
mid_step        = 0.02
extent_low      = 0.02
extent_high     = 0.401
extent_step     = 0.02

# find out how long the database will need to be so we can reserve the whole array in memory up front.
# this will speed up the process
for mid in np.round(np.arange(mid_low, mid_high, mid_step), 2):
    for extent in np.round(np.arange(extent_low, extent_high, extent_step), 2):
        for amp_1 in np.round(np.arange(.05, .96, .05), 2):
            num_records += 1
for mid in np.round(np.arange(mid_low, mid_high, mid_step), 2):
    for extent in np.round(np.arange(extent_low, extent_high, extent_step), 2):
        loc_1 = np.round(mid - extent, 2)
        loc_3 = np.round(mid + extent, 2)
        for spot in np.round(np.arange(.02, extent*2-.01, .02), 2):
            for amp_1 in np.round(np.arange(.05, .91, .05), 2):
                for amp_2 in np.round(np.arange(.05, .96-amp_1, .05), 2):
                    num_records += 1

print(num_records)

SampleRate  = 100
x           = np.round(np.linspace(-2, 2, 4*int(SampleRate), endpoint=False), 3)
data_array  = np.zeros((num_records, len(cols)))
gmm_array   = np.zeros((num_records, len(gmm_names)))
start       = time.time()
idx         = 0

# create the database
# two deltas loop
for mid in np.round(np.arange(mid_low, mid_high, mid_step), 2):
    for extent in np.round(np.arange(extent_low, extent_high, extent_step), 2):
        loc_1 = np.round(mid - extent, 2)
        loc_2 = np.round(mid + extent, 2)
        for amp_1 in np.round(np.arange(.05, .96, .05), 2):
            amp_2           = np.round(1.0 - amp_1, 2)
            delta           = sfl.Delta(x, loc1=loc_1, loc2=loc_2, amp1=amp_1, amp2=amp_2)
            ems             = mfl.Moments(x, delta.scatter)
            vals            = [loc_1, loc_2, .49, amp_1, amp_2, 0.]
            data_array[idx] = np.concatenate((vals, ems.moments))
            target_moments  = [ems.mean, ems.variance, ems.std, ems.skew, ems.kurtosis]
            results, _, _   = gmm.match_moments(target_moments)
            gmm_scatter     = gmm.gmm_pdf(x, results)
            gmm_ems         = mfl.Moments(x, gmm_scatter)
            gmm_vals        = np.concatenate((vals, results))
            gmm_array[idx]  = np.concatenate((gmm_vals, gmm_ems.moments))
            idx += 1
            if idx % 1000 == 0:
                print(idx)
# three deltas loop
for mid in np.round(np.arange(mid_low, mid_high, mid_step), 2):
    for extent in np.round(np.arange(extent_low, extent_high, extent_step), 2):
        loc_1 = np.round(mid - extent, 2)
        loc_3 = np.round(mid + extent, 2)
        for spot in np.round(np.arange(.02, extent*2-.01, .02), 2):
            loc_2 = np.round(spot + loc_1, 2)
            for amp_1 in np.round(np.arange(.05, .91, .05), 2):
                for amp_2 in np.round(np.arange(.05, .96-amp_1, .05), 2):
                    amp_3           = np.round(1.0 - amp_1 - amp_2, 2)
                    delta           = sfl.Delta(x, loc1=loc_1, loc2=loc_2, loc3=loc_3,
                                                amp1=amp_1, amp2=amp_2, amp3=amp_3)
                    ems             = mfl.Moments(x, delta.scatter)
                    vals            = [loc_1, loc_2, loc_3, amp_1, amp_2, amp_3]
                    data_array[idx] = np.concatenate((vals, ems.moments))
                    target_moments  = [ems.mean, ems.variance, ems.std, ems.skew, ems.kurtosis]
                    results, _, _   = gmm.match_moments(target_moments)
                    gmm_scatter     = gmm.gmm_pdf(x, results)
                    gmm_ems         = mfl.Moments(x, gmm_scatter)
                    gmm_vals        = np.concatenate((vals, results))
                    gmm_array[idx]  = np.concatenate((gmm_vals, gmm_ems.moments))
                    idx += 1
                    if idx % 1000 == 0:
                        print(idx)

database    = pd.DataFrame(data_array, columns=cols)
database2   = pd.DataFrame(gmm_array, columns=gmm_names)
end         = time.time()
print("Time taken: ", (end - start)/60)

folder = '/your/folder/here/'
database.to_csv(folder + 'delta_database.csv', index=False)
database2.to_csv(folder + 'gmm_database.csv', index=False)

##### clean up gmm database

In [ ]:
folder  = '/your/folder/here/'
db      = pd.read_csv(folder + 'delta_database.csv')
db2     = pd.read_csv(folder + 'gmm_database.csv')

In [ ]:
db2['good']     = True

SampleRate      = 100
ScatterLen      = 4
x               = np.round(np.linspace(-ScatterLen/2, ScatterLen/2, ScatterLen*SampleRate, endpoint=False), 3)

for idx, row in db2.iterrows():
    if idx % 1000 == 0:
        print(idx)
    gmm_scatter = sfl.GMM(x, row['mu1'], row['mu2'], row['sigma'], row['w1'])
    mnts        = mfl.Moments(gmm_scatter.x, gmm_scatter.scatter)
    mnts_short  = [mnts.mean, mnts.std, mnts.skew, mnts.kurtosis]

    if (
        (np.round(mnts.mean, 2) != np.round(row['mean'], 2))
        or (np.round(mnts.std, 2) != np.round(row['std'], 2))
        or (np.round(mnts.skew, 2) != np.round(row['skew'], 2))
        or (np.round(mnts.kurtosis, 2) != np.round(row['kurtosis'], 2))
        or (np.round(db2.loc[idx, 'mean'], 2) != np.round(db.loc[idx, 'mean'], 2))
        or (np.round(db2.loc[idx, 'std'], 2) != np.round(db.loc[idx, 'std'], 2))
        or (np.round(db2.loc[idx, 'skew'], 2) != np.round(db.loc[idx, 'skew'], 2))
        or (np.round(db2.loc[idx, 'kurtosis'], 2) != np.round(db.loc[idx, 'kurtosis'], 2))
    ):
        db2.loc[idx, 'good'] = False

db2     = db2[db2['good'] == True]

print(db.shape)
print(db2.shape)

In [ ]:
db2.to_csv(folder + 'gmm_database.csv', index=False)